In [1]:
import numpy as np
import tensorflow as tf
import keras

In [ ]:
# 1. 데이터 불러오기:
(train_input, train_target), _ = keras.datasets.fashion_mnist.load_data()
# _ : "시험 문제(Test data)" 
# -> 지금은 공부(Train)만 할 거라서 일단 서랍에 넣어두고 안 쓰겠다. (파이썬의 관습적인 표현)

# 2. 데이터 상태 확인하기
print(train_input.shape)   # (60000, 28, 28)
# 가로 28픽셀, 세로 28픽셀짜리 흑백 이미지 파일이 총 60,000장 들어있음
print(train_input.dtype)   # uint8
# 부호가 없는 8비트 정수형(Unsigned Integer 8-bit) -> 컴퓨터가 이미지를 숫자로 다루고 있다는 뜻
print(train_input.min(), train_input.max()) # 0 ~ 255
# 이미지의 각 픽셀은 밝기에 따라 0(완전 검은색) ~ 255(완전 흰색) 사이의 숫자로 이루어져 있음

(60000, 28, 28)
uint8
0 255


In [ ]:
# 3. 랜덤 시드 고정 : 실험의 '재현성'을 위해 컴퓨터의 무작위 주사위 번호를 42번으로 고정
keras.utils.set_random_seed(42)

# 4. 인공신경망 모델 만들기
inputs = keras.layers.Input(shape=(784,))   # 입력층(Input Layer)을 만듦
# 왜 784일까? : 원래 이미지는 (28, 28)인 2차원 바둑판 모양, Dense 층은 1차원만 받을 수 있음
# 28 x 28 = 784 -> "이미지를 한 줄로 길게 이어 붙인 784개의 숫자 형태로 뇌에 집어넣겠다"라고 선언
dense  = keras.layers.Dense(10, activation='softmax')   # 출력층(Output Layer)을 만듦
# 10: 우리가 맞추어야 할 옷의 종류(티셔츠, 바지, 구두 등)가 총 10가지이기 때문에 뉴런을 10개 배치
# activation='softmax' : 소프트맥스 활성화 함수
# -> 10개의 뉴런이 각각 점수를 내놓을 때, 이를 확률(총합이 100% 또는 1이 되는 값)로 변환해주는 수학 공식
model  = keras.Sequential([inputs, dense])
# 위에서 만든 입력 통로(inputs)와 정답을 맞추는 뉴런들(dense)을 차례대로 일렬로 연결 -> 모델(model)로 조립

# 5. 설계도 확인: model.summary()
model.summary()
# 'Output Shape: (None, 10)' : 한번에 몇 개의 데이터를 넣든(None은 배치 크기 유연성을 의미), 최종 출력은 10개의 확률 값으로 나온다는 뜻
# 'Param # : 7850' : 컴퓨터가 앞으로 학습하며 스스로 찾아내야 할 가중치와 편향의 총 개수
# 입력 특성 784개 x 출력 뉴런 10개 = 7,840개의 가중치(w)
# + 10개의 편향(b) -> 7840 + 10 = 7850개

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │         7,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,850 (30.66 KB)

 Trainable params: 7,850 (30.66 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# [1단계] 데이터 전처리 (0~1 사이로 압축 및 1차원으로 펴기)
# 1. 0~255 사이의 픽셀 값을 0~1 사이로 정규화 (나누기 255)
train_scaled = train_input / 255.0

# 2. (60000, 28, 28) 크기를 (60000, 784) 크기로 1차원 변형
train_scaled = train_scaled.reshape(-1, 784)

print("전처리 후 데이터 크기:", train_scaled.shape)

전처리 후 데이터 크기: (60000, 784)


In [ ]:
# [2단계] 모델 설계
keras.utils.set_random_seed(42)

# 모델 조립
model = keras.Sequential([
    keras.layers.Input(shape=(784,)),
    keras.layers.Dense(10, activation='softmax')
])

In [ ]:
# [3단계] 모델 컴파일
model.compile(
    optimizer='adam',      # 오답 노트를 보고 가중치를 똑똑하게 수정하는 알고리즘 (가장 대중적)
    loss='sparse_categorical_crossentropy', # 다중 분류(10개 중 하나 맞추기)에서 사용하는 채점 공식
    metrics=['accuracy']   # 평가 지표 : 정확도
)

In [ ]:
# [4단계] 모델 학습 (fit)
model.fit(train_scaled, train_target, epochs=5) # epochs=5 : 5번 반복

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 792us/step - accuracy: 0.7995 - loss: 0.5976
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 765us/step - accuracy: 0.8422 - loss: 0.4606
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 770us/step - accuracy: 0.8505 - loss: 0.4348
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 777us/step - accuracy: 0.8549 - loss: 0.4212
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 1s 768us/step - accuracy: 0.8576 - loss: 0.4124
